[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S22_flujo_competencia.ipynb)

# Sesión 22 · Flujo de competencia

**Módulo 5: Machine Learning** · ⏱️ Duración estimada: 60 a 90 minutos en el notebook, más el trabajo del proyecto (P3) y la publicación (Post 2)

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Estimar el desempeño de un modelo con validación cruzada honesta, sin fugas.
2. Preparar un archivo de envío con el formato exacto y revisarlo antes de enviarlo.
3. Llevar un registro de experimentos que permita comparar y reproducir.
4. Explicar por qué perseguir el leaderboard público lleva al sobreajuste.
5. Elegir la entrega final con validación cruzada y cerrar el modelo de tu proyecto.

## 📋 Qué debes saber antes
Sesiones 17 a 21: validación cruzada, AUC, Random Forest, LightGBM, `Pipeline` y `ColumnTransformer`.

## 🧭 Cómo trabajar este notebook
Hoy simulas una **competencia de modelos**, como las de las plataformas de ciencia de datos:
- Tienes `clientes_train`, con la respuesta, y `clientes_test`, **sin** la respuesta.
- Entregas un archivo con una probabilidad por cliente de `clientes_test`. La función `evaluar_envio` te devuelve el **puntaje público**: el AUC sobre un 30 % del test.
- El puntaje que decide la competencia es el **privado**: el AUC sobre el otro 70 %, que se conoce al cierre. Aquí lo verás con `revelar_privado`, pero úsalo solo cuando el enunciado lo pida.

Ejecuta las celdas **en orden** con **Shift + Enter**, escribe tu código debajo de `# Tu código aquí` y verifica. Algunas celdas tardan unos segundos. Los archivos que guardes quedan en la carpeta de trabajo (en Colab, el panel 📁 de la izquierda).

Después del notebook vienen el 🧱 **avance del proyecto** (P3 completo) y la guía del 📣 **Post 2**.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Prepara la competencia, aplica el estilo de gráficos y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.** Tampoco busques las respuestas ocultas del test: arruinarías la simulación.

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara la competencia simulada, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import os
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})

# ---------- Competencia simulada: ¿el cliente contratará el depósito a plazo? ----------
_n = 4000
_edad = rng.integers(18, 76, _n)
_saldo = np.round(rng.lognormal(7.5, 1.0, _n), 2)
_mov = rng.poisson(12, _n)
_ant = rng.integers(1, 241, _n)
_canal = rng.choice(["app", "web", "agencia"], _n, p=[0.5, 0.2, 0.3])
_seg = rng.choice(["A", "B", "C"], _n, p=[0.2, 0.5, 0.3])
_cont = rng.poisson(1, _n)
_tarj = (rng.random(_n) < 0.4).astype(int)
_logit = -4.2 + 1.8 * (0.6 * np.log(_saldo / 1800) + 0.04 * (_mov - 12) + 0.8 * (_canal == "app") - 0.5 * _tarj
                       + 0.5 * _cont - 0.12 * _cont ** 2 + 0.0012 * (_edad - 45) ** 2 + 0.9 * ((_seg == "A") & (_canal == "app")))
_y = (rng.random(_n) < 1 / (1 + np.exp(-_logit))).astype(int)
_dur = np.round(rng.gamma(2, 60, _n) + _y * rng.gamma(3, 120, _n))
_todo = pd.DataFrame({
    "id": rng.permutation(np.arange(100001, 100001 + _n)), "edad": _edad, "saldo_promedio": _saldo,
    "movimientos_mes": _mov, "antiguedad_meses": _ant, "canal": _canal, "segmento": _seg,
    "contactos_previos": _cont, "tiene_tarjeta": _tarj, "duracion_llamada": _dur, "acepta": _y,
})
clientes_train = _todo.iloc[:2000].reset_index(drop=True)
clientes_test = _todo.iloc[2000:].drop(columns="acepta").reset_index(drop=True)
# Etiquetas ocultas del test (no las mires: arruinarían la simulación) y división pública / privada.
_ETIQUETAS = dict(zip(_todo["id"].iloc[2000:], _todo["acepta"].iloc[2000:]))
_ES_PUBLICO = dict(zip(clientes_test["id"], rng.random(2000) < 0.3))
del _todo
_ENVIOS = [0]

_D = copy.deepcopy({"clientes_train": clientes_train, "clientes_test": clientes_test})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _auc(real, prob):
    """AUC con la fórmula de rangos de Mann-Whitney (sin scikit-learn); los empates reciben el rango promedio."""
    real, prob = [int(v) for v in real], [float(v) for v in prob]
    orden = sorted(range(len(prob)), key=lambda i: prob[i])
    rangos = [0.0] * len(prob)
    i = 0
    while i < len(orden):
        j = i
        while j + 1 < len(orden) and prob[orden[j + 1]] == prob[orden[i]]:
            j += 1
        for k in range(i, j + 1):
            rangos[orden[k]] = (i + j) / 2 + 1
        i = j + 1
    pos = [rangos[i] for i in range(len(real)) if real[i] == 1]
    n_pos, n_neg = len(pos), len(real) - len(pos)
    return (math.fsum(pos) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def _particion():
    """El X_train, X_test, y_train, y_test del alumno, si existen y son coherentes."""
    v = [globals().get(n) for n in ("X_train", "X_test", "y_train", "y_test")]
    if all(isinstance(a, pd.DataFrame) for a in v[:2]) and all(isinstance(a, pd.Series) for a in v[2:]) and len(v[0]) == len(v[2]) and len(v[1]) == len(v[3]):
        return v
    return None


def _auc_modelo(m, X, y):
    return _auc(y.tolist(), m.predict_proba(X)[:, 1])


def _problemas(envio):
    if not isinstance(envio, pd.DataFrame):
        return ["el envío debería ser un DataFrame o la ruta de un CSV"]
    if list(envio.columns) != ["id", "prob_acepta"]:
        return ["las columnas deberían ser exactamente id y prob_acepta"]
    p = []
    if len(envio) != len(clientes_test):
        p.append(f"tiene {len(envio)} filas y se esperaban {len(clientes_test)}")
    if envio["id"].duplicated().any():
        p.append("hay ids repetidos")
    if set(envio["id"]) != set(clientes_test["id"]):
        p.append("los ids no coinciden con los de clientes_test")
    prob = pd.to_numeric(envio["prob_acepta"], errors="coerce")
    if prob.isna().any():
        p.append("hay probabilidades vacías o que no son números")
    elif ((prob < 0) | (prob > 1)).any():
        p.append("hay probabilidades fuera del rango de 0 a 1")
    return p


def _leer(envio):
    if isinstance(envio, str):
        if not os.path.exists(envio):
            raise FileNotFoundError(f"No encuentro el archivo {envio!r}.")
        envio = pd.read_csv(envio)
    p = _problemas(envio)
    if p:
        raise ValueError("Envío rechazado: " + "; ".join(p) + ".")
    return dict(zip(envio["id"], envio["prob_acepta"].astype(float)))


def _puntaje(envio, publico):
    prob = _leer(envio)
    ids = [i for i in clientes_test["id"] if _ES_PUBLICO[i] == publico]
    return round(_auc([_ETIQUETAS[i] for i in ids], [prob[i] for i in ids]), 4)


def evaluar_envio(envio, mostrar=True):
    """Puntaje público (AUC sobre ~30 % del test). Acepta la ruta de un CSV o un DataFrame con id y prob_acepta."""
    puntaje = _puntaje(envio, True)
    _ENVIOS[0] += 1
    if mostrar:
        print(f"📊 Envío n.º {_ENVIOS[0]} · puntaje público (AUC): {puntaje:.4f}")
    return puntaje


def revelar_privado(envio):
    """Puntaje privado (AUC sobre el ~70 % restante). En una competencia real solo lo verías al cierre."""
    puntaje = _puntaje(envio, False)
    print(f"🔒 Puntaje privado (AUC): {puntaje:.4f}")
    return puntaje


def _pipe_ref(modelo):
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    num = [c for c in globals().get("columnas_num", [])]
    cat = [c for c in globals().get("columnas_cat", [])]
    return Pipeline([("prep", ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])),
                     ("modelo", modelo)])


def _modelos_ref():
    from lightgbm import LGBMClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.linear_model import LogisticRegression
    return {"base": LogisticRegression(max_iter=1000),
            "rf": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1),
            "lgbm": LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=4, min_child_samples=30, random_state=42, verbose=-1)}


def _pliegues():
    from sklearn.model_selection import StratifiedKFold
    return list(StratifiedKFold(5, shuffle=True, random_state=42).split(clientes_train, clientes_train["acepta"]))


def _notas_cv(pipe):
    from sklearn.base import clone
    X, y = clientes_train[columnas_modelo], clientes_train["acepta"]
    notas = []
    for tr, va in _pliegues():
        m = clone(pipe).fit(X.iloc[tr], y.iloc[tr])
        notas.append(_auc(y.iloc[va].tolist(), m.predict_proba(X.iloc[va])[:, 1]))
    return notas


def _oof(pipe):
    from sklearn.base import clone
    X, y = clientes_train[columnas_modelo], clientes_train["acepta"]
    oof = np.zeros(len(X))
    for tr, va in _pliegues():
        oof[va] = clone(pipe).fit(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[va])[:, 1]
    return oof


def _prob_test(pipe):
    from sklearn.base import clone
    m = clone(pipe).fit(clientes_train[columnas_modelo], clientes_train["acepta"])
    return m.predict_proba(clientes_test[columnas_modelo])[:, 1]


def _es_fuga(c):
    return _h(c) == "b371b065a517973894768c4c87ba27f6bf7ba25160599d3be223f62110ff57bf"


def _columnas_ok():
    cn, cc = globals().get("columnas_num"), globals().get("columnas_cat")
    cm = globals().get("columnas_modelo")
    return isinstance(cn, list) and isinstance(cc, list) and isinstance(cm, list) and cm == cn + cc and len(cm) == 8 \
        and not {"id", "acepta"} & set(cm) and not any(_es_fuga(c) for c in cm) and set(cm) <= set(clientes_train.columns)


def _revisar_envio_df(r, nombre, prob_ref, tol=1e-9, motivo="revisa el modelo y que uses `predict_proba(...)[:, 1]`"):
    v = r.var(nombre)
    if v is _FALTA:
        return False
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame.")
        return False
    p = _problemas(v)
    if p:
        r.mal(f"`{nombre}` no tiene el formato de envío: {p[0]}.")
        return False
    if v["id"].tolist() != clientes_test["id"].tolist():
        r.mal(f"Los ids de `{nombre}` deberían ir en el mismo orden que en `clientes_test`.")
        return False
    if not np.allclose(v["prob_acepta"].to_numpy(float), prob_ref, atol=tol):
        r.mal(f"Las probabilidades de `{nombre}` no coinciden con las esperadas; {motivo}.")
        return False
    r.ok(f"`{nombre}` es correcto.")
    return True


def _revisar_archivo(r, ruta, df):
    if not os.path.exists(ruta):
        r.mal(f"No encuentro el archivo `{ruta}`. Guárdalo con `to_csv(\"{ruta}\", index=False)`.")
        return
    leido = pd.read_csv(ruta)
    if list(leido.columns) != list(df.columns) or len(leido) != len(df):
        r.mal(f"`{ruta}` no tiene las mismas columnas o filas que el DataFrame. ¿Usaste `index=False`?")
    else:
        r.ok(f"`{ruta}` se guardó bien.")


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    _sin_cambios_df(r, "clientes_train", "clientes_test")
    cn, cc, cm = r.var("columnas_num", list), r.var("columnas_cat", list), r.var("columnas_modelo", list)
    if any(v is _FALTA for v in (cn, cc, cm)):
        r.fin()
        return
    todas = cn + cc
    if cm != todas:
        r.mal("`columnas_modelo` debería ser `columnas_num + columnas_cat`.")
    elif [c for c in todas if c not in clientes_train.columns]:
        r.mal(f"Estas columnas no existen en `clientes_train`: {[c for c in todas if c not in clientes_train.columns]}.")
    elif any(_es_fuga(c) for c in todas):
        r.mal(f"`{[c for c in todas if _es_fuga(c)][0]}` solo se conoce **después** de llamar al cliente: es una fuga. Quítala.")
    elif "acepta" in todas:
        r.mal("`acepta` es lo que quieres predecir: no puede ser una variable del modelo.")
    elif "id" in todas:
        r.mal("`id` es solo un identificador: no describe al cliente y no debe entrar al modelo.")
    elif [c for c in cn if not pd.api.types.is_numeric_dtype(clientes_train[c])]:
        r.mal(f"En `columnas_num` hay columnas de texto: {[c for c in cn if not pd.api.types.is_numeric_dtype(clientes_train[c])]}.")
    elif [c for c in cc if pd.api.types.is_numeric_dtype(clientes_train[c])]:
        r.mal(f"En `columnas_cat` hay columnas numéricas: {[c for c in cc if pd.api.types.is_numeric_dtype(clientes_train[c])]}.")
    elif len(todas) != 8 or len(set(todas)) != len(todas):
        r.mal(f"Se esperaban 8 columnas utilizables, sin repetir, y hay {len(set(todas))}.")
    else:
        r.ok("Elegiste bien las columnas del modelo.")
    pipe = r.var("pipe_base")
    if pipe is not _FALTA and _columnas_ok():
        if type(pipe).__name__ != "Pipeline" or list(pipe.named_steps) != ["prep", "modelo"] or type(pipe.named_steps["modelo"]).__name__ != "LogisticRegression":
            r.mal("`pipe_base` debería ser un `Pipeline` con los pasos \"prep\" y \"modelo\" (una `LogisticRegression`).")
        else:
            ref = _notas_cv(pipe)
            v = r.var("notas_cv")
            if v is not _FALTA:
                v = np.asarray(v, dtype=float)
                if v.shape != (5,):
                    r.mal(f"`notas_cv` tiene forma {v.shape} y se esperaban 5 notas, una por pliegue.")
                elif not np.allclose(v, ref, atol=1e-6):
                    r.mal("`notas_cv` no coincide: usa `cross_val_score` con `cv` (el `StratifiedKFold` pedido) y `scoring=\"roc_auc\"`.")
                else:
                    r.ok("`notas_cv` es correcto.")
            _esc(r, "auc_cv_base", statistics.fmean(ref), "el promedio de `notas_cv`", tol=1e-6)
            _esc(r, "desv_cv_base", statistics.pstdev(ref), "la desviación estándar de `notas_cv` (`notas_cv.std()`)", tol=1e-6)
            if hasattr(pipe.named_steps["modelo"], "classes_"):
                _esc(r, "auc_train_base", _auc(clientes_train["acepta"].tolist(), _prob_train(pipe)), "el AUC de `pipe_base`, entrenado con todo `clientes_train`, sobre ese mismo conjunto", tol=1e-6)
            else:
                r.mal("Entrena `pipe_base` con todo `clientes_train` antes de calcular `auc_train_base`.")
    elif pipe is not _FALTA:
        r.mal("Corrige primero las columnas para revisar `pipe_base`.")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_mas_optimista": "0155bde5382b25a72510c0c24ffd2447d92c2770a6ac05a7be7c732186dd263e",
    })
    r.fin()


def _prob_train(pipe):
    from sklearn.base import clone
    m = clone(pipe).fit(clientes_train[columnas_modelo], clientes_train["acepta"])
    return m.predict_proba(clientes_train[columnas_modelo])[:, 1]


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    f = r.funcion("problemas_envio")
    if f is not _FALTA:
        ids = clientes_test["id"]
        bueno = pd.DataFrame({"id": ids, "prob_acepta": np.linspace(0, 1, len(ids))})
        casos = [("un envío correcto (con probabilidades 0 y 1 en los extremos)", bueno, True),
                 ("un envío correcto con las filas en otro orden", bueno.iloc[::-1].reset_index(drop=True), True),
                 ("un envío sin la columna prob_acepta", bueno.rename(columns={"prob_acepta": "prob"}), False),
                 ("un envío vacío", bueno.iloc[0:0], False),
                 ("un envío al que le falta una fila", bueno.iloc[1:], False),
                 ("un envío con una probabilidad vacía", bueno.assign(prob_acepta=bueno["prob_acepta"].where(bueno.index != 5)), False),
                 ("un envío con una probabilidad de 1.2", bueno.assign(prob_acepta=bueno["prob_acepta"].where(bueno.index != 7, 1.2)), False),
                 ("un envío con una probabilidad negativa", bueno.assign(prob_acepta=bueno["prob_acepta"].where(bueno.index != 9, -0.1)), False),
                 ("un envío con un id repetido", bueno.assign(id=bueno["id"].where(bueno.index != 0, bueno["id"].iloc[1])), False),
                 ("un envío con un id que no está en el test", bueno.assign(id=bueno["id"].where(bueno.index != 0, 1)), False)]
        for texto, df, valido in casos:
            try:
                res = f(df.copy(), ids.copy())
            except Exception as ex:
                r.mal(f"`problemas_envio` con {texto} lanzó {type(ex).__name__}: {ex}")
                continue
            if not isinstance(res, list):
                r.mal(f"`problemas_envio` debería devolver una lista y devolvió {type(res).__name__}.")
                break
            if valido and res:
                r.mal(f"`problemas_envio` marcó problemas en {texto}, y es válido.")
            elif not valido and not res:
                r.mal(f"`problemas_envio` no detectó ningún problema en {texto}.")
            else:
                r.ok(f"`problemas_envio` funciona con {texto}.")
    pipe = globals().get("pipe_base")
    if _columnas_ok() and hasattr(pipe, "named_steps"):
        if _revisar_envio_df(r, "envio_base", _prob_test(pipe)):
            _revisar_archivo(r, "envio_base.csv", envio_base)
            if os.path.exists("envio_base.csv"):
                _esc(r, "publico_base", _puntaje("envio_base.csv", True), "el resultado de `evaluar_envio(\"envio_base.csv\")`", tol=1e-9)
    else:
        r.mal("Primero resuelve el ejercicio 1.")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_orden_importa": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    r.funcion("correr_experimento")
    reg = r.var("registro")
    columnas = ["experimento", "modelo", "n_variables", "auc_cv", "auc_publico", "notas"]
    if reg is not _FALTA and not _columnas_ok():
        r.mal("Primero resuelve el ejercicio 1.")
    elif reg is not _FALTA:
        if not isinstance(reg, pd.DataFrame):
            r.mal(f"`registro` es de tipo {type(reg).__name__} y se esperaba un DataFrame.")
        elif list(reg.columns) != columnas:
            r.mal(f"Las columnas de `registro` deberían ser {columnas}, en ese orden.")
        elif len(reg) < 3:
            r.mal(f"`registro` tiene {len(reg)} filas y se esperaban al menos 3 experimentos.")
        elif reg["experimento"].duplicated().any():
            r.mal("En `registro` hay nombres de experimento repetidos: cada uno debe ser único.")
        elif [e for e in ("base", "rf", "lgbm") if e not in set(reg["experimento"])]:
            r.mal(f"Faltan en `registro` los experimentos {[e for e in ('base', 'rf', 'lgbm') if e not in set(reg['experimento'])]}.")
        else:
            bien = True
            refs = _modelos_ref()
            for _, fila in reg.iterrows():
                exp = fila["experimento"]
                ruta = f"envio_{exp}.csv"
                if not os.path.exists(ruta):
                    r.mal(f"No encuentro `{ruta}`: `correr_experimento` debería guardar el envío de cada experimento.")
                    bien = False
                    continue
                try:
                    pub = _puntaje(ruta, True)
                except ValueError as ex:
                    r.mal(f"`{ruta}`: {ex}")
                    bien = False
                    continue
                if not _es_numero(fila["auc_publico"]) or abs(float(fila["auc_publico"]) - pub) > 1e-9:
                    r.mal(f"En `registro`, el `auc_publico` de \"{exp}\" no coincide con el puntaje de `{ruta}`.")
                    bien = False
                if not _es_numero(fila["auc_cv"]) or not 0.5 < float(fila["auc_cv"]) < 1:
                    r.mal(f"En `registro`, el `auc_cv` de \"{exp}\" debería ser un AUC entre 0,5 y 1.")
                    bien = False
                elif exp in refs:
                    ref = statistics.fmean(_notas_cv(_pipe_ref(refs[exp])))
                    if abs(float(fila["auc_cv"]) - ref) > 5e-5:
                        r.mal(f"El `auc_cv` de \"{exp}\" no coincide con el modelo pedido para ese experimento; revisa sus hiperparámetros y el `cv`.")
                        bien = False
                if fila["n_variables"] != len(columnas_modelo):
                    r.mal(f"En `registro`, `n_variables` de \"{exp}\" debería ser la cantidad de columnas del modelo.")
                    bien = False
                if not isinstance(fila["notas"], str) or not fila["notas"].strip():
                    r.mal(f"Escribe una nota breve para el experimento \"{exp}\": qué probaste o qué esperabas.")
                    bien = False
            if bien:
                r.ok("`registro` está completo y coincide con los archivos de envío.")
            _revisar_archivo(r, "registro_experimentos.csv", reg)
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_anotar_fallos": "6c5fb3b25e6ba7dcf12155440e0c51b36a0492e24123968a10f8c31c304ebb6e",
    })
    r.fin()


def _ruido(prob, semilla):
    return np.clip(prob + np.random.default_rng(semilla).normal(0, 0.01, len(prob)), 0, 1)


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    reg = globals().get("registro")
    if not isinstance(reg, pd.DataFrame) or "auc_cv" not in reg or "experimento" not in reg:
        r.mal("Primero resuelve el ejercicio 3.")
        r.fin()
        return
    top = str(reg.loc[reg["auc_cv"].astype(float).idxmax(), "experimento"])
    me = r.var("mejor_exp", str)
    if me is not _FALTA:
        if me != top:
            r.mal("`mejor_exp` debería ser el experimento con mayor `auc_cv` en `registro`.")
        else:
            r.ok("`mejor_exp` es correcto.")
    ruta = f"envio_{top}.csv"
    if not os.path.exists(ruta):
        r.mal(f"No encuentro `{ruta}`.")
        r.fin()
        return
    base = pd.read_csv(ruta)["prob_acepta"].to_numpy(float)
    ids = pd.read_csv(ruta)["id"]
    pm = r.var("prob_mejor")
    if pm is not _FALTA:
        pm = np.asarray(pm, dtype=float)
        if pm.shape != base.shape or not np.allclose(pm, base, atol=1e-12):
            r.mal(f"`prob_mejor` debería ser la columna `prob_acepta` de `{ruta}`.")
        else:
            r.ok("`prob_mejor` es correcto.")
    pubs = [_puntaje(pd.DataFrame({"id": ids, "prob_acepta": _ruido(base, s)}), True) for s in range(50)]
    _ser(r, "publicos", pubs, "el puntaje público de cada una de las 50 versiones con ruido, con la semilla como índice", indice=list(range(50)), tol=1e-9)
    st = pubs.index(max(pubs))
    _esc(r, "semilla_top", st, "la semilla con mayor puntaje público", tol=0)
    if r.var("envio_trampa") is not _FALTA:
        et = envio_trampa
        if not isinstance(et, pd.DataFrame) or list(et.columns) != ["id", "prob_acepta"] or et["id"].tolist() != ids.tolist() \
                or not np.allclose(et["prob_acepta"].to_numpy(float), _ruido(base, st), atol=1e-12):
            r.mal("`envio_trampa` debería tener los ids y la versión con ruido de la semilla ganadora.")
        else:
            r.ok("`envio_trampa` es correcto.")
    _esc(r, "publico_mejor", _puntaje(ruta, True), "el puntaje público del mejor experimento", tol=1e-9)
    _esc(r, "privado_mejor", _puntaje(ruta, False), "el puntaje privado del mejor experimento", tol=1e-9)
    _esc(r, "privado_trampa", _puntaje(pd.DataFrame({"id": ids, "prob_acepta": _ruido(base, st)}), False), "el puntaje privado de `envio_trampa`", tol=1e-9)
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_trampa_gana_privado": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    if not _columnas_ok():
        r.mal("Primero resuelve el ejercicio 1.")
        r.fin()
        return
    refs = _modelos_ref()
    oofs = {"rf": _oof(_pipe_ref(refs["rf"])), "lgbm": _oof(_pipe_ref(refs["lgbm"]))}
    oofs["mezcla"] = (oofs["rf"] + oofs["lgbm"]) / 2
    for k in ("rf", "lgbm", "mezcla"):
        v = r.var(f"oof_{k}")
        if v is _FALTA:
            continue
        v = np.asarray(v, dtype=float)
        if v.shape != oofs[k].shape:
            r.mal(f"`oof_{k}` tiene forma {v.shape} y se esperaba {oofs[k].shape}: una probabilidad por cliente de `clientes_train`.")
        elif not np.allclose(v, oofs[k], atol=1e-9):
            r.mal(f"`oof_{k}` no coincide: usa `cross_val_predict` con el mismo `cv` y `method=\"predict_proba\"`." if k != "mezcla"
                  else "`oof_mezcla` debería ser el promedio de `oof_rf` y `oof_lgbm`.")
        else:
            r.ok(f"`oof_{k}` es correcto.")
    y = clientes_train["acepta"].tolist()
    aucs = {k: _auc(y, oofs[k]) for k in ("rf", "lgbm", "mezcla")}
    _ser(r, "auc_oof", [aucs[k] for k in ("rf", "lgbm", "mezcla")], "el AUC de cada predicción fuera de pliegue, con índice rf, lgbm y mezcla", indice=["rf", "lgbm", "mezcla"], tol=1e-9)
    gana = max(aucs, key=aucs.get)
    if gana == "mezcla" and all(os.path.exists(f"envio_{k}.csv") for k in ("rf", "lgbm")):
        prob = (pd.read_csv("envio_rf.csv")["prob_acepta"].to_numpy(float) + pd.read_csv("envio_lgbm.csv")["prob_acepta"].to_numpy(float)) / 2
    elif os.path.exists(f"envio_{gana}.csv"):
        prob = pd.read_csv(f"envio_{gana}.csv")["prob_acepta"].to_numpy(float)
    else:
        r.mal("Faltan los archivos de envío del ejercicio 3.")
        r.fin()
        return
    if _revisar_envio_df(r, "envio_final", prob, tol=1e-9, motivo="deberían ser las de la opción con mayor `auc_oof`"):
        _revisar_archivo(r, "envio_final.csv", envio_final)
        if os.path.exists("envio_final.csv"):
            _esc(r, "privado_final", _puntaje("envio_final.csv", False), "el puntaje privado de `envio_final.csv`", tol=1e-9)
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    if not _columnas_ok():
        r.mal("Primero resuelve el ejercicio 1.")
        r.fin()
        return
    from lightgbm import LGBMClassifier
    probs = [_prob_test(_pipe_ref(LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=4, min_child_samples=30, subsample=0.8,
                                                  subsample_freq=1, colsample_bytree=0.8, random_state=s, verbose=-1))) for s in range(5)]
    if _revisar_envio_df(r, "envio_semillas", np.mean(probs, axis=0), tol=1e-9):
        _esc(r, "publico_semillas", _puntaje(envio_semillas, True), "el puntaje público de `envio_semillas`", tol=1e-9)
    r.fin()


print("✅ Setup listo. Competencia preparada, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
Un banco llamó a sus clientes para ofrecerles un **depósito a plazo**. Por cada cliente tienes: `id`, edad, saldo promedio (en soles), movimientos al mes, meses de antigüedad, canal preferido, segmento, contactos de campañas anteriores, si tiene tarjeta de crédito, la **duración de la llamada** (en segundos) y, solo en `clientes_train`, si **aceptó** (1) o no (0).

Funciones de la competencia: `evaluar_envio(envio)` y `revelar_privado(envio)`. Ambas aceptan la ruta de un CSV o un DataFrame con las columnas `id` y `prob_acepta`, y rechazan el envío si el formato no es el correcto. `evaluar_envio(envio, mostrar=False)` no imprime nada.

In [ ]:
print(clientes_train.head(), "\n")
print(clientes_train.shape, clientes_test.shape)
print(clientes_train["acepta"].mean().round(3))
print(clientes_train.groupby("acepta")["duracion_llamada"].mean().round(0))

---
## 1. Validación honesta

### 📘 Concepto
En una competencia, el conjunto de prueba es de los organizadores. Tu única forma honesta de saber si un cambio mejora el modelo es la **validación cruzada** sobre el entrenamiento. Para que sea honesta:
- **Solo variables que existirán al predecir.** Un dato que se registra después del evento es una fuga (sesión 21).
- **Todo el preprocesamiento dentro de un `Pipeline`**, para que se reajuste en cada pliegue.
- **Los mismos pliegues para todos los modelos** (`StratifiedKFold` con semilla fija): así las diferencias vienen de los modelos y no del azar de la partición.
- **Mira también la dispersión** entre pliegues: si dos modelos difieren menos que esa variación, la diferencia puede ser ruido.

El AUC en los mismos datos con los que entrenaste siempre es optimista: el modelo ya vio esas respuestas.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

gen_ej = np.random.default_rng(3)
X_ej = pd.DataFrame({"x": gen_ej.normal(size=400)})
y_ej = pd.Series((X_ej["x"] + gen_ej.normal(scale=1.5, size=400) > 1).astype(int))
notas_ej = cross_val_score(LogisticRegression(), X_ej, y_ej, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc")
print(notas_ej.round(3), round(notas_ej.mean(), 3), round(notas_ej.std(), 3))

### ✍️ Tu turno · Ejercicio 1: la línea base honesta
**Parte A.**
1. `columnas_num` y `columnas_cat`: listas con las columnas numéricas y de texto que **se pueden usar** para predecir antes de llamar al cliente. `columnas_modelo = columnas_num + columnas_cat`.
2. `cv = StratifiedKFold(5, shuffle=True, random_state=42)`: úsalo en toda la sesión.
3. `pipe_base`: un `Pipeline` con `("prep", ColumnTransformer(...))` (`StandardScaler` para `columnas_num` y `OneHotEncoder(handle_unknown="ignore")` para `columnas_cat`) y `("modelo", LogisticRegression(max_iter=1000))`.
4. `notas_cv`: el resultado de `cross_val_score` de `pipe_base` sobre `clientes_train[columnas_modelo]` con `cv` y `scoring="roc_auc"`; `auc_cv_base` y `desv_cv_base`: su promedio y su desviación estándar.
5. Entrena `pipe_base` con todo `clientes_train`; `auc_train_base`: su AUC en ese mismo conjunto.

**Parte B.** Responde en `pred_mas_optimista` con `"entrenamiento"` o `"cv"`: ¿cuál de los dos AUC es más optimista?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Pregúntate por cada columna: ¿la conozco **antes** de llamar al cliente? ¿Describe al cliente o solo lo identifica?
</details>

<details><summary>💡 Pista 2</summary>

Es el pipeline de la sesión 21 con otras listas de columnas. `notas_cv.mean()` y `notas_cv.std()` dan el promedio y la desviación.
</details>

---
## 2. El archivo de envío

### 📘 Concepto
Las competencias piden un archivo con un formato exacto; si no lo cumple, se rechaza o, peor, se evalúa mal sin avisar. El formato de hoy:
- un CSV con **exactamente** las columnas `id` y `prob_acepta`;
- **una fila por cliente** de `clientes_test`, sin ids repetidos ni ids ajenos;
- probabilidades numéricas entre 0 y 1, sin vacíos.

Se entregan **probabilidades** (`predict_proba`), no clases: el AUC mide el orden. Y se guarda con `index=False`, para no agregar una columna de índice. Antes de gastar un envío conviene tener tu propia función que revise el formato.

In [ ]:
envio_ej = pd.DataFrame({"id": [7, 8, 9], "prob_acepta": [0.10, 0.85, 0.40]})
envio_ej.to_csv("envio_ejemplo.csv", index=False)
print(open("envio_ejemplo.csv").read())
print(envio_ej["id"].duplicated().any(), envio_ej["prob_acepta"].between(0, 1).all(), envio_ej["prob_acepta"].isna().any())

### ✍️ Tu turno · Ejercicio 2: prepara y revisa tu envío
**Parte A.**
1. `problemas_envio(envio, ids)`: una función que devuelve una **lista** de textos con los problemas de `envio` (vacía si está bien), comparando con la Series `ids` de ids esperados. Debe detectar: columnas distintas de `id` y `prob_acepta`, cantidad de filas distinta de la de `ids`, ids repetidos o que no están en `ids`, probabilidades vacías y probabilidades fuera de 0 a 1. El orden de las filas **no** es un problema.
2. `envio_base`: un DataFrame con el `id` de `clientes_test` (en el mismo orden) y `prob_acepta`, la probabilidad de aceptar según `pipe_base`.
3. Revisa que `problemas_envio(envio_base, clientes_test["id"])` esté vacía, guarda `envio_base.csv` sin índice y `publico_base = evaluar_envio("envio_base.csv")`.

**Parte B.** La función de evaluación une tu archivo con las respuestas por `id`. Responde en `pred_orden_importa` con `"sí"` o `"no"`: ¿cambiaría el puntaje si las filas del archivo estuvieran en otro orden?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Empieza con `problemas = []` y agrega un texto por cada condición que falle. Si las columnas no son las correctas, devuelve enseguida: las demás revisiones las necesitan.
</details>

<details><summary>💡 Pista 2</summary>

Útiles: `list(envio.columns)`, `envio["id"].duplicated().any()`, `set(envio["id"]) == set(ids)`, `envio["prob_acepta"].isna().any()` y `envio["prob_acepta"].between(0, 1).all()`.
</details>

---
## 3. Registro de experimentos

### 📘 Concepto
Después de diez pruebas nadie recuerda qué dio 0,81 y qué dio 0,79. Un **registro de experimentos** es una tabla con una fila por intento: nombre, modelo, variables, AUC de validación cruzada, puntaje público y una nota con qué cambiaste y por qué. Sirve para:
- elegir la entrega final con evidencia y no de memoria;
- reproducir cualquier resultado;
- contar tu proceso en el README o en una publicación.

Anota **también** lo que salió peor: saber qué no funciona evita repetirlo. Una función que corre un experimento completo y devuelve su fila hace que registrar sea automático.

In [ ]:
def probar_ej(nombre, modelo):
    notas = cross_val_score(modelo, X_ej, y_ej, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc")
    return {"experimento": nombre, "modelo": type(modelo).__name__, "auc_cv": notas.mean()}

from sklearn.dummy import DummyClassifier

registro_ej = pd.DataFrame([probar_ej("logistica", LogisticRegression()), probar_ej("trivial", DummyClassifier())])
print(registro_ej)

### ✍️ Tu turno · Ejercicio 3: tres experimentos registrados
**Parte A.**
1. `correr_experimento(nombre, modelo, notas)`: una función que arma un pipeline con el mismo preprocesamiento de `pipe_base` y `modelo`, calcula su AUC promedio de validación cruzada con `cv`, lo entrena con todo `clientes_train`, guarda su envío en `f"envio_{nombre}.csv"`, lo evalúa con `evaluar_envio` y devuelve un diccionario con `experimento`, `modelo` (el nombre de la clase, `type(modelo).__name__`), `n_variables`, `auc_cv`, `auc_publico` y `notas`.
2. Corre tres experimentos:
   - `"base"`: `LogisticRegression(max_iter=1000)`;
   - `"rf"`: `RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)`;
   - `"lgbm"`: `LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=4, min_child_samples=30, random_state=42, verbose=-1)`.
3. `registro`: un DataFrame con una fila por experimento y esas seis columnas, en ese orden. Guárdalo en `registro_experimentos.csv` sin índice.

¿El orden por `auc_cv` coincide con el orden por `auc_publico`?

**Parte B.** Responde en `pred_anotar_fallos` con `"sí"` o `"no"`: ¿conviene dejar en el registro los experimentos que dieron peor resultado?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Dentro de la función: crea un `ColumnTransformer` nuevo (no reutilices el de `pipe_base`), arma el `Pipeline` y reutiliza lo que escribiste en los ejercicios 1 y 2.
</details>

<details><summary>💡 Pista 2</summary>

`registro = pd.DataFrame([correr_experimento("base", ...), correr_experimento("rf", ...), correr_experimento("lgbm", ...)])`.
</details>

---
## 4. Leaderboard público vs. privado

### 📘 Concepto
El puntaje público se calcula con pocas filas, así que tiene **ruido**: un envío puede subir en el público por suerte. Si eliges tus envíos mirando el público, terminas ajustando el modelo a ese 30 % del test: es sobreajuste, pero al leaderboard. Al cierre, el privado lo revela y muchos participantes caen varios puestos (en inglés se le llama *shake-up*).

La defensa es la de siempre: **decide con validación cruzada**, que usa más datos, y trata el público como un control más, no como la meta.

En este ejercicio lo compruebas con una "trampa": sumas un ruido pequeño al mejor envío, pruebas 50 versiones y te quedas con la que más sube en el público. El ruido no puede contener información sobre los clientes. Fíjate además en el contador de envíos: en las plataformas reales hay un límite diario justamente para frenar esta práctica.

In [ ]:
prob_ej = np.array([0.2, 0.5, 0.9])
for semilla in range(3):
    print(semilla, np.clip(prob_ej + np.random.default_rng(semilla).normal(0, 0.01, len(prob_ej)), 0, 1).round(4))

### ✍️ Tu turno · Ejercicio 4: la trampa del leaderboard
**Parte A.**
1. `mejor_exp`: el nombre del experimento de `registro` con mayor `auc_cv`, y `prob_mejor`: la columna `prob_acepta` de su archivo `f"envio_{mejor_exp}.csv"`, como array.
2. `publicos`: una Series con el puntaje público (usa `mostrar=False`) de 50 versiones con ruido, una por `semilla` de `range(50)`, con la semilla como índice. Cada versión es `np.clip(prob_mejor + np.random.default_rng(semilla).normal(0, 0.01, len(prob_mejor)), 0, 1)`.
3. `semilla_top`: la semilla con mayor puntaje público, y `envio_trampa`: un DataFrame de envío con los ids de `clientes_test` y esa versión.
4. `publico_mejor`: el puntaje público del archivo del mejor experimento. Compáralo con `publicos.max()`.
5. Ahora sí, revela: `privado_mejor = revelar_privado(f"envio_{mejor_exp}.csv")` y `privado_trampa = revelar_privado(envio_trampa)`.

**Parte B.** Antes de revelar, predice: responde en `pred_trampa_gana_privado` con `"sí"` o `"no"`: ¿`envio_trampa` le ganará en el privado al envío original?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

`registro.loc[registro["auc_cv"].idxmax(), "experimento"]` da el nombre. Para `publicos`, arma un diccionario `{semilla: puntaje}` dentro de un `for` y conviértelo en Series.
</details>

<details><summary>💡 Pista 2</summary>

`publicos.idxmax()` da la semilla. El DataFrame de la trampa es `pd.DataFrame({"id": clientes_test["id"], "prob_acepta": version_ganadora})`.
</details>

---
## 🏋️ Reto final: la entrega final, decidida con validación cruzada
Una forma frecuente de ganar un poco es **mezclar** modelos: promediar sus probabilidades. Para decidir si la mezcla conviene sin mirar el público, usa predicciones **fuera de pliegue**: `cross_val_predict` devuelve, para cada cliente de `clientes_train`, la probabilidad que le dio el modelo entrenado sin él.

1. `oof_rf` y `oof_lgbm`: `cross_val_predict(pipeline, clientes_train[columnas_modelo], clientes_train["acepta"], cv=cv, method="predict_proba")[:, 1]` para los pipelines de los experimentos `"rf"` y `"lgbm"`. `oof_mezcla`: su promedio.
2. `auc_oof`: una Series con el AUC de las tres, con índice `"rf"`, `"lgbm"` y `"mezcla"`.
3. `envio_final`: el envío de la opción con mayor `auc_oof`. Si gana `"mezcla"`, promedia las probabilidades de `envio_rf.csv` y `envio_lgbm.csv`. Guárdalo en `envio_final.csv` sin índice y agrégalo como una fila más a `registro`.
4. Recién entonces: `privado_final = revelar_privado("envio_final.csv")`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Necesitas los pipelines del rf y del lgbm fuera de `correr_experimento`: arma cada uno otra vez, con un `ColumnTransformer` nuevo.
</details>

<details><summary>💡 Pista 2</summary>

`auc_oof = pd.Series({"rf": roc_auc_score(y, oof_rf), "lgbm": ..., "mezcla": ...})` y `auc_oof.idxmax()` dice qué entregar.
</details>

---
## 🚀 Nivel pro (opcional): promediar semillas
Si un modelo usa azar al entrenar (por ejemplo, cada árbol de LightGBM ve un 80 % de las filas y de las columnas), cambiar la semilla cambia un poco sus predicciones. Promediar varias semillas reduce esa variación sin agregar información nueva.

Crea `envio_semillas`: un envío con el promedio de las probabilidades de 5 pipelines con `LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=4, min_child_samples=30, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=semilla, verbose=-1)`, para `semilla` de `range(5)`, entrenados con todo `clientes_train`. Guarda `publico_semillas = evaluar_envio(envio_semillas)`. ¿Lo agregarías al registro? ¿Con qué AUC de validación cruzada?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## 🧱 Avance del proyecto: P3 completo · un modelo evaluado sin fugas

**Qué hacer**
1. **Elige la variable a predecir.** Usa la pregunta 5 del proyecto: ¿qué factores anticipan que un caso se resuelva a favor del consumidor? Si tus datos no tienen una variable así (o tiene muy pocos casos), usa el **respaldo Bank Marketing**:
   - UCI Machine Learning Repository, dataset id 222, archivo `bank-additional-full.csv` (41 188 filas, separador `;`), variable objetivo `y`;
   - cita: Moro, S., Rita, P., & Cortez, P. (2014). *Bank Marketing* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5K306;
   - revisa en su página la licencia vigente y escríbela en el README;
   - **excluye `duration`**: solo se conoce después de la llamada, la misma trampa que evitaste hoy en el ejercicio 1.
2. En `notebooks/04_modelo.ipynb`, separa un conjunto de prueba estratificado **antes** de explorar para modelar, y no lo toques hasta el final.
3. Para cada columna, pregúntate si existiría al momento de decidir. Anota en el notebook las que excluyes por fuga y por qué.
4. Arma un `Pipeline` con `ColumnTransformer` y compara con la misma validación cruzada al menos: un modelo trivial (`DummyClassifier`), una regresión logística y un modelo de árboles (Random Forest o LightGBM).
5. Las clases están desbalanceadas (en Bank Marketing, cerca de 11 % de "yes"): reporta AUC y, con el umbral que elijas según costos, la matriz de confusión, la precisión y la sensibilidad (sesión 19). No reportes solo exactitud.
6. Lleva un **registro de experimentos** (`reports/registro_experimentos.csv`) con al menos 5 filas, incluidas las que no funcionaron.
7. Al final, evalúa **una sola vez** el modelo elegido en el conjunto de prueba, y guarda en `reports/figures/` la curva ROC y las importancias de las variables.
8. Actualiza el README con una sección **Modelo**: qué predice, con qué datos, cómo se evaluó, el resultado en prueba frente al modelo trivial, y sus límites (por ejemplo, importancia no es causalidad).

**Por qué lo haría un analista**
Un modelo solo aporta si su evaluación es creíble. Mostrar la línea base, la validación sin fugas y el registro de lo que se probó es lo que convence a quien tiene que decidir si usarlo, y lo que te permite defenderlo en una entrevista.

**Cómo debe verse el resultado**
Un `04_modelo.ipynb` que se ejecuta de principio a fin, un registro de experimentos que cuenta el proceso, dos o tres figuras que se entienden solas y un README en el que alguien sin conocimientos técnicos entiende qué predice el modelo, cuánto mejor lo hace que no tener modelo y para qué serviría.

---
## 📣 Post 2: publica tu modelo

Esta publicación cuenta **qué problema resuelve tu modelo y cómo sabes que funciona**, no qué librerías usaste.

**Estructura sugerida**
1. **Gancho** (1 o 2 líneas): la pregunta de negocio y el resultado principal en palabras simples.
2. **Datos**: qué usaste, con la fuente y la licencia (si es Bank Marketing, la cita completa).
3. **Cómo lo evaluaste**: la comparación con una línea base, la validación cruzada y qué variable excluiste por fuga. Es lo que muestra criterio.
4. **Qué aporta**: una traducción al negocio, con supuestos explícitos (por ejemplo, "llamando al 20 % con mayor probabilidad se alcanzaría a X de cada 10 interesados").
5. **Una limitación** honesta y el **enlace** al repositorio, con una pregunta abierta.

**Imágenes**: la curva ROC o un gráfico de "con modelo vs. sin modelo" y las importancias de las variables. Míralas en el celular antes de publicar.

**Antes de publicar, revisa que:**
- [ ] cada cifra coincide con tu notebook y con tu registro de experimentos;
- [ ] citas la fuente de los datos y su licencia;
- [ ] explicas qué variable excluiste por fuga y por qué;
- [ ] comparas contra una línea base y no presentas la exactitud sola;
- [ ] no hablas de causas: el modelo muestra asociaciones;
- [ ] agregaste un texto alternativo a cada imagen;
- [ ] el repositorio es público y el README está al día.

Escríbelo con tu voz: cuenta qué decisión te costó más y qué aprendiste de ella.

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar por qué la validación cruzada es la guía y el test público no.
- [ ] Reconocer una variable que solo existe después del evento y excluirla.
- [ ] Preparar un archivo de envío con el formato exacto y revisarlo con una función.
- [ ] Llevar un registro de experimentos y usarlo para decidir.
- [ ] Explicar qué es el *shake-up* entre el leaderboard público y el privado.
- [ ] Decidir si una mezcla de modelos conviene usando predicciones fuera de pliegue.

**Próximo módulo (S23–S27):** SQL, segmentación y dashboard. Antes de empezar, termina P3 y publica el Post 2.